# 1. Explore the data and train the bearing fault model

**Story:** a planer motor in a sawmill. We want to spot **imbalance** and **bearing wear** early, from one second of vibration plus temperature and motor current.

This notebook: simulate sensor data, look at what faults look like, compute features, train a small neural network, and evaluate it.

In [ ]:
# Install the demo package from the cloned repo (skipped if already installed)
import importlib.util
if importlib.util.find_spec("bearing") is None:
    %pip install -q -e "..[notebook]"
    print("Installed. If imports fail, restart the kernel once.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bearing.config import CLASSES, FEATURE_NAMES, SAMPLE_RATE_HZ
from bearing.simulate import simulate_window, make_dataset
from bearing.features import extract_features, as_dict

## What does a failing bearing look like?
Same motor, three conditions. Bearing wear shows up as sharp periodic impacts that ring a high-frequency resonance.

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(3, 2, figsize=(12, 7))
for row, (mode, sev) in enumerate([("healthy", 0), ("imbalance", 0.8), ("bearing_wear", 0.8)]):
    w = simulate_window(mode, sev, rng, shaft_hz=30)
    t = np.arange(w["samples"].size) / SAMPLE_RATE_HZ
    axes[row, 0].plot(t[:1024], w["samples"][:1024], lw=0.7)
    axes[row, 0].set_title(f"{mode}: waveform (first 0.2 s)"); axes[row, 0].set_ylabel("g")
    spec = np.abs(np.fft.rfft(w["samples"])) / w["samples"].size
    f = np.fft.rfftfreq(w["samples"].size, 1 / SAMPLE_RATE_HZ)
    axes[row, 1].semilogy(f, spec + 1e-6, lw=0.6); axes[row, 1].set_title(f"{mode}: spectrum")
    axes[row, 1].set_xlabel("Hz")
    print(mode, as_dict(extract_features(w["samples"], w["temp_c"], w["current_a"])))
plt.tight_layout(); plt.show()

## Build a training set
600 one-second windows per class. Fault severity is random, so mild faults look almost healthy, which is what makes early detection hard.

In [ ]:
X, y = make_dataset(n_per_class=600, seed=7)
print(X.shape, "features:", FEATURE_NAMES)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, c in enumerate(CLASSES):
    ax[0].scatter(X[y == i, 4], X[y == i, 7], s=6, label=c)
    ax[1].hist(X[y == i, 3], bins=40, alpha=0.6, label=c)
ax[0].set_xlabel("1x band (g)"); ax[0].set_ylabel("bearing resonance band (g)"); ax[0].legend()
ax[1].set_xlabel("kurtosis"); ax[1].legend(); plt.show()

## Train and evaluate
A small scaler + neural network. Small on purpose: it has to run in milliseconds on a mill edge server.

In [ ]:
from bearing.model import train
clf, metrics = train(X, y, seed=7)
print(metrics["report"])
print("confusion matrix (rows = truth):", CLASSES)
print(np.array(metrics["confusion_matrix"]))

In [ ]:
# Save for notebook 2
import pickle, os, json
os.makedirs("../models", exist_ok=True)
with open("../models/bearing-sklearn.pkl", "wb") as f:
    pickle.dump(clf, f)
with open("../models/bearing-metrics.json", "w") as f:
    json.dump({k: v for k, v in metrics.items() if k != "report"}, f, indent=2)
print("saved ../models/bearing-sklearn.pkl")

**Talking point:** the mistakes are mild imbalance called healthy. That is the business trade-off to discuss with maintenance: alert earlier and accept more false alarms, or wait for more evidence.